# Titanic Survival Prediction
This notebook demonstrates a complete machine learning workflow to predict passenger survival on the Titanic using Logistic Regression.

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

## 1. Data Loading and Initial Exploration
First, we load the dataset and take a look at a few sample rows to understand the features we are working with.

In [20]:
df = pd.read_csv("titanic_data_updated.csv")

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
612,613,yes,third,"Murphy, Miss. Margaret Jane",female,NaN,1,0,367230,15.50,NaN,Q
252,253,no,first,"Stead, Mr. William Thomas",male,62.0,0,0,113514,26.55,C87,S
678,679,no,third,"Goodwin, Mrs. Frederick (Augusta Tyler)",female,43.0,1,6,CA 2144,46.90,NaN,S
880,881,yes,second,"Shelley, Mrs. William (Imanita Parrish Hall)",female,25.0,0,1,230433,26.00,NaN,S
447,448,yes,first,"Seward, Mr. Frederic Kimber",male,34.0,0,0,113794,26.55,NaN,S


In [21]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering
We create new features like `Family_Size` and extract the `Deck` from the `Cabin` column to provide more meaningful information to the model.

In [22]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]

df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
722,723,no,second,"Gillespie, Mr. William Henry",male,34.0,0,0,12233,13.0000,Missing,S,1,M
662,663,no,first,"Colley, Mr. Edward Pomeroy",male,47.0,0,0,5727,25.5875,E58,S,1,E
259,260,yes,second,"Parrish, Mrs. (Lutie Davis)",female,50.0,0,1,230433,26.0000,Missing,S,2,M
699,700,no,third,"Humblen, Mr. Adolf Mathias Nicolai Olsen",male,42.0,0,0,348121,7.6500,F G63,S,1,F
601,602,no,third,"Slabenoff, Mr. Petco",male,NaN,0,0,349214,7.8958,Missing,S,1,M
515,516,no,first,"Walker, Mr. William Anderson",male,47.0,0,0,36967,34.0208,D46,S,1,D


In [23]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [24]:
X = df.drop('Survived', axis=1)
y = df['Survived']

## 3. Data Splitting
We split the data into training (to teach the model) and testing (to evaluate the model) sets. This ensures we can test the model on data it hasn't seen before.

In [25]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [26]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


## 4. Outlier Handling and Data Cleaning
We use Z-scores to remove extreme outliers in `Age` and 'clip' the `Fare` column to prevent extreme values from distorting our model's learning.

In [27]:
y_test

565     no
160     no
553    yes
860     no
241    yes
      ... 
880    yes
91      no
883     no
473    yes
637     no
Name: Survived, Length: 179, dtype: object

In [28]:
# Age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Zscore'] = (X_train['Age']-mean_age) / std_age

musk = (abs(X_train['Age']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# Fare

fare_q1 = X_train['Fare'].quantile(0.25)
fare_q3 = X_train['Fare'].quantile(0.75)

IQR = fare_q3 - fare_q1

minimum = max(0, fare_q1-1.5*IQR)
maximum = fare_q3-1.5*IQR

X_train['Fare'] = X_train['Fare'].clip(minimum,maximum)

C:\Users\User\AppData\Local\Temp\ipykernel_17052\3482228902.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum,maximum)


## 5. Building Preprocessing Pipelines
We create different pipelines for numerical and categorical data. This handles missing values (imputation) and scales the data so all features are on a similar range.

In [31]:
# pipeline
# numerical column

p1 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]
)

In [32]:
categories = [['third','second','first']]

In [35]:
# categorical column

p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore')),
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=categories)),
        ('scaler', MinMaxScaler())
    ]
)

In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ]
)

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])